# M2 · Pengaruh resolusi terhadap deteksi objek kecil
Apakah native 1280 memperbaiki AP-small dibanding native 640? Kedua model dilatih sebelumnya pada pasangan sumber yang sama selama lima epoch. Notebook ini menjalankan ulang inferensi dari kedua checkpoint pada 356 gambar validasi dan membandingkan hasil dengan acuan lokal.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.210', 'pycocotools'], check=True)

In [ ]:
import csv, json, tempfile, tarfile
from pathlib import Path
import torch
files = list(Path('/kaggle/input').rglob('validation_e2.tar.gz'))
assert len(files) <= 1, files
project = Path(tempfile.mkdtemp(prefix='ciringoni-e2-'))
if files:
    with tarfile.open(files[0]) as archive:
        archive.extractall(project, filter='data')
else:
    import shutil
    manifests = list(Path('/kaggle/input').rglob('geometry.json'))
    assert len(manifests) == 1, manifests
    shutil.copytree(manifests[0].parent, project, dirs_exist_ok=True)
gt = json.loads((project/'hasil_m1/coco_val_gt.json').read_text())
assert (len(gt['images']), len(gt['annotations'])) == (356, 2205)
device = '0' if torch.cuda.is_available() else 'cpu'
output = Path('/kaggle/working')
print('Validasi:', len(gt['images']), 'gambar;', len(gt['annotations']), 'kotak. Perangkat:', device)

## Gambar → prediksi → skor
Evaluator mencocokkan ID validasi, memetakan kotak native ke acuan 640, lalu menghitung AP-small COCO serta presisi dan recall pada confidence 0,05 dan IoU 0,50. mAP50–95 berasal dari validator Ultralytics.

In [ ]:
rows = []
for size, run in [(640,'E2_control_native_640'), (1280,'E2_native_1280')]:
    data = project/str(size)
    yaml = data/'data.yaml'
    yaml.write_text(f'path: {data}\ntrain: images/val\nval: images/val\nnames: [infected]\n')
    dest = output/run
    command = [sys.executable, str(project/'hasil_m2/evaluasi_e2.py'), str(data/'best.pt'), str(yaml), str(project/'geometry.json'), str(dest), '--size', str(size), '--device', device]
    subprocess.run(command, check=True)
    rows.append(json.loads((dest/'metrik.json').read_text()))
    print(run, 'AP-small', rows[-1]['AP-small'])

In [ ]:
with (project/'hasil_m2/evaluasi/eksperimen_e2_val.csv').open() as stream:
    baseline = {r['id']: r for r in csv.DictReader(stream)}
for r in rows:
    r['perangkat'] = 'GPU' if device == '0' else 'CPU'
    r['ap_small_acuan'] = float(baseline[r['id']]['AP-small'])
    r['selisih_dari_acuan'] = r['AP-small'] - r['ap_small_acuan']
with (output/'hasil_e2_kaggle.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print('Perubahan AP-small 1280 terhadap 640:', rows[1]['AP-small'] - rows[0]['AP-small'])
print('CSV dan prediksi per resolusi tersedia di Output.')